# CivicCare — Feature Engineering, Target Leakage Audit & Representation Design
**Project:** Problem Statement 11 — Crowdsourced Civic Issue Reporting and Resolution Analytics Dashboard  
**Team Role:** Member 1 — Machine Learning & Data Acquisition Pipeline  
**Focus:** Feature Extraction, Spurious Correlation Resolution, and Feature Isolation across all 3 Models

---

## Executive Overview
Machine learning models are only as robust as their feature representations. In municipal civic issue prediction, naive feature engineering leads to two catastrophic failure modes:
1. **Target Label Leakage:** Feeding parent administrative labels directly into text inputs, causing models to artificially score 100% while failing completely on real citizen queries.
2. **Spurious Location Correlation:** Models associating common urban spatial words (like *"road"*, *"street"*, *"cross"*) exclusively with pothole complaints, failing when a citizen reports a broken streetlight or overflowing sewer on a road.

This notebook documents how we systematically audited and eliminated target leakage, engineered subword text representations, stabilized skewed resolution times, and extracted linguistic markers from municipal officer remarks across all three models.


In [1]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

sys.path.append(os.path.abspath('..'))
from ml.config import MASTER_CATEGORIES, NYC_CATEGORY_MAPPING, BBMP_CATEGORY_MAPPING, INDIAN_MUNICIPAL_SLAS

print("Feature engineering modules and taxonomy definitions loaded successfully.")


Feature engineering modules and taxonomy definitions loaded successfully.


## 1. Target Label Leakage Audit & Elimination

### The Problem We Diagnosed
In early prototype iterations, our team observed an uncanny test accuracy of **1.0000 (100%)**. In applied machine learning, an accuracy of 100% on complex human text is almost always an indicator of **Target Label Leakage**.

Upon inspecting the preprocessing pipeline, we found the root cause:
- The input string was constructed as:
  $$\text{input\_text} = \text{complaint\_type} + \text{" "} + \text{descriptor}$$
- However, our target variable `standard_category` was directly mapped from `complaint_type`!
- Consequently, the model was simply reading the ground truth category embedded inside the input text rather than understanding the civic problem.

### The Systematic Fix
We strictly stripped all parent administrative classifications (`complaint_type`, department names, agency codes). The feature pipeline now strictly takes **only** the citizen's grievance text (`descriptor`).


In [2]:
# Demonstration: Testing for Target Leakage in Input Features
sample_raw_complaint = {
    "complaint_type": "Street Condition",
    "descriptor": "Pothole in middle of driving lane",
    "target_category": "Roads & Infrastructure"
}

# LEAKED INPUT (Flawed):
leaked_input = f"{sample_raw_complaint['complaint_type']} {sample_raw_complaint['descriptor']}"

# LEAKAGE-FREE INPUT (Our Method):
clean_input = sample_raw_complaint['descriptor']

print(f"Leaked Feature  : '{leaked_input}' (Contains parent category clue 'Street Condition')")
print(f"Clean Feature   : '{clean_input}' (Pure citizen complaint description)")
print(f"Is target in clean feature? {'Roads' in clean_input or 'Infrastructure' in clean_input} -> NO LEAKAGE!")


Leaked Feature  : 'Street Condition Pothole in middle of driving lane' (Contains parent category clue 'Street Condition')
Clean Feature   : 'Pothole in middle of driving lane' (Pure citizen complaint description)
Is target in clean feature? False -> NO LEAKAGE!


## 2. Feature Engineering for Model 1: Civic Category Classification

### Why Word-Level TF-IDF Alone Is Insufficient
Standard word-level TF-IDF treats words as independent atomic symbols. In Indian municipal governance:
1. Citizens use localized terms and acronyms: *SWD* (Storm Water Drain), *nalah*, *khata*, *BESCOM*, *raja kaluve*.
2. Complaints frequently contain spelling typos, plurals, and compounding (*street-light*, *streetlight*, *potholes*, *crater*).

### Why We Chose a FeatureUnion of Word & Character Subwords
We designed a composite feature extractor combining:
1. **Word n-grams $(1, 3)$ with sublinear scaling:** Captures multi-word civic concepts (*"overflowing manhole"*, *"low voltage"*, *"illegal dumping"*).
2. **Character subword n-grams $(3, 5)$ with word-boundary awareness (`char_wb`):** Breaks words into morphological n-grams (e.g., *"trans"*, *"ansfo"*, *"sform"* for *transformer*). Even if an exact word has a typo or varied suffix, the subword n-grams preserve semantic similarity.


In [3]:
# Constructing the FeatureUnion
feature_extractor = FeatureUnion([
    ('word_tfidf', TfidfVectorizer(ngram_range=(1, 3), max_features=25000, sublinear_tf=True)),
    ('char_tfidf', TfidfVectorizer(ngram_range=(3, 5), analyzer='char_wb', max_features=35000, sublinear_tf=True))
])

sample_texts = [
    "pothole on driving lane",
    "severely damaged pot-holes on road",
    "overflowing stormwater drain nalah"
]

feature_extractor.fit(sample_texts)
feat_matrix = feature_extractor.transform(sample_texts)
print(f"Transformed feature matrix shape: {feat_matrix.shape} (Sparse CSR Matrix)")
print(f"Word vocabulary size: {len(feature_extractor.transformer_list[0][1].vocabulary_)}")
print(f"Character subword vocabulary size: {len(feature_extractor.transformer_list[1][1].vocabulary_)}")


Transformed feature matrix shape: (3, 222) (Sparse CSR Matrix)
Word vocabulary size: 32
Character subword vocabulary size: 190


### Resolving the Spurious Location Correlation
**The Discovery:**
When auditing early models against real web complaints, we observed that complaints like:
- *"Complete blackout on 100 feet road for past 3 nights"*
- *"Stray cattle roaming on high speed ring road"*
- *"Footpath completely blocked by illegal stalls along MG road"*
were ALL being misclassified as **`Roads & Infrastructure`**.

**The Cause:**
In municipal datasets, the word **`"road"`** had a massive positive coefficient ($+2.35$) solely for `Roads & Infrastructure`, while being negative for all other 8 categories. Because citizens naturally mention where a problem is located (*"on 100 feet road"*), the word `"road"` acted as a gravity well!

**The Solution: Location-Invariance Engineering**
We engineered location-invariant training samples by ensuring common urban location phrases (*"on main road"*, *"along 80 feet road"*, *"in 4th cross lane"*, *"near metro station"*) appear **uniformly across all 9 categories**. This forces the model to ignore location tokens and focus on the true problem keywords (*blackout*, *stray cattle*, *illegal stalls*).


In [4]:
# Demonstration of Location-Invariance Augmentation
LOCATION_CONTEXTS = [
    'on main road', 'near ring road', 'in 4th cross lane', 'along 80 feet road',
    'near metro station', 'in our locality', 'near market circle', 'outside apartment gate'
]

sample_concepts = [
    ("street light not working", "Street Lighting & Electrical"),
    ("garbage accumulation", "Solid Waste Management"),
    ("overflowing sewage", "Drainage & Water Supply")
]

augmented_samples = []
for concept, cat in sample_concepts:
    for loc in LOCATION_CONTEXTS[:3]:
        augmented_samples.append((f"{concept} {loc}", cat))

print(f"Generated {len(augmented_samples)} location-invariant training phrases:")
for text, cat in augmented_samples:
    print(f"  [{cat:30s}] -> '{text}'")


Generated 9 location-invariant training phrases:
  [Street Lighting & Electrical  ] -> 'street light not working on main road'
  [Street Lighting & Electrical  ] -> 'street light not working near ring road'
  [Street Lighting & Electrical  ] -> 'street light not working in 4th cross lane'
  [Solid Waste Management        ] -> 'garbage accumulation on main road'
  [Solid Waste Management        ] -> 'garbage accumulation near ring road'
  [Solid Waste Management        ] -> 'garbage accumulation in 4th cross lane'
  [Drainage & Water Supply       ] -> 'overflowing sewage on main road'
  [Drainage & Water Supply       ] -> 'overflowing sewage near ring road'
  [Drainage & Water Supply       ] -> 'overflowing sewage in 4th cross lane'


## 3. Feature Engineering for Model 2: Resolution Time & SLA Engine

### Why Continuous Linear Regression Fails
As established in our EDA, municipal resolution turnaround hours span from 0.1 hours to 10,000+ hours with extreme right skewness. We formulated a **dual-head target strategy**:

1. **Classification Target (SLA Compliance):**
   A binary indicator of whether a complaint is resolved within the maximum tolerable SLA hours defined by the Indian Municipal Citizen Charter:
   $$y_{\text{SLA}} = \mathbb{I}\left(\text{actual\_hours} \le \text{SLA}_{\max}(\text{category})\right)$$
2. **Regression Target (Continuous Turnaround):**
   To stabilize variance and eliminate outlier gradient explosion, we apply a log-transformation:
   $$y_{\text{log}} = \log(1 + \text{actual\_hours})$$

### Tabular Input Features
We extracted:
- `category` (One-hot encoded)
- `agency` (One-hot encoded)
- `borough` (One-hot encoded)
- `created_hour` (Continuous 0–23)
- `created_day_of_week` (Continuous 0–6)
- `is_weekend` (Binary 0/1)


In [5]:
preprocessor_sla = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), ['category', 'agency', 'borough']),
        ('num', 'passthrough', ['created_hour', 'created_day_of_week', 'is_weekend'])
    ]
)

sample_sla_df = pd.DataFrame([
    {'category': 'Roads & Infrastructure', 'agency': 'DOT', 'borough': 'QUEENS', 'created_hour': 14, 'created_day_of_week': 2, 'is_weekend': 0},
    {'category': 'Drainage & Water Supply', 'agency': 'DEP', 'borough': 'BROOKLYN', 'created_hour': 8, 'created_day_of_week': 6, 'is_weekend': 1}
])

transformed = preprocessor_sla.fit_transform(sample_sla_df)
print(f"Transformed tabular feature vector shape: {transformed.shape}")
print("Preprocessed feature matrix successfully initialized.")


Transformed tabular feature vector shape: (2, 9)
Preprocessed feature matrix successfully initialized.


## 4. Feature Engineering for Model 3: Officer Response Quality & Fake Closure Verifier

### Why Linguistic & Metadata Features in Officer Remarks?
Municipal field officers facing high ticket quotas often employ canned, repetitive remarks to close complaints prematurely. We engineered specialized behavioral features from `staff_remarks`:
1. `remarks_len`: Character length of the closure remark.
2. `word_count`: Number of words.
3. `has_canned_word`: Binary flag identifying minimal single-character or lazy remarks (`"."`, `"-"`, `"close"`, `"closed"`, `"attended"`, `"clear"`).
4. `has_promise_word`: Binary flag detecting deferred future promises made while simultaneously marking tickets closed (*"action will be taken"*, *"forwarded task"*, *"shortly"*).
5. `cat_freq` & `channel_freq`: Statistical prior probabilities of closure by category and filing channel.


In [6]:
def extract_officer_features(remarks_series, category_series, channel_series):
    remarks = remarks_series.fillna("").astype(str).str.lower().str.strip()
    
    df_feat = pd.DataFrame()
    df_feat['remarks_len'] = remarks.str.len()
    df_feat['word_count'] = remarks.str.split().str.len()
    df_feat['has_canned_word'] = remarks.isin([".", "-", "close", "closed", "attend", "attended", "clear"]).astype(int)
    df_feat['has_promise_word'] = remarks.str.contains(r"action|shortly|process|taken", regex=True).astype(int)
    return df_feat

sample_remarks = pd.Series([
    "Pothole filled with wet mix macadam and compacted with roller",
    ".",
    "Action will be taken shortly"
])

features_demo = extract_officer_features(sample_remarks, None, None)
features_demo['raw_text'] = sample_remarks
features_demo


,remarks_len,word_count,has_canned_word,has_promise_word,raw_text
0,61,10,0,0,Pothole filled with wet mix macadam and compac...
1,1,1,1,0,.
2,28,5,0,1,Action will be taken shortly


## 5. Summary & Hand-off to Model Training

Through this feature engineering phase:
1. **Target Leakage:** Completely eradicated by restricting NLP inputs to citizen grievance text.
2. **Subword Robustness:** Designed Word $(1, 3)$ + Char $(3, 5)$ FeatureUnion to tolerate municipal acronyms, typos, and compounding.
3. **Location-Invariance:** Neutralized spurious spatial correlations (e.g. `"road"`) across all 9 categories.
4. **Variance Stabilization:** Established log-transformation and Citizen Charter SLA thresholds for turnaround prediction.
5. **Behavioral Indicators:** Formulated linguistic markers to detect canned premature ticket closures.

**Next Notebook:** `03_Model_Selection_Ensemble_and_Evaluation.ipynb` — Training candidate algorithms, ensembling, and evaluating on holdouts and live web data.
